# Posture Detection V2 — 01 Dataset Protocol and Temporal Setup

V2 extends the completed frame-based system into a temporal 3D posture-analysis project centered on MotionBERT. This notebook establishes the dataset before model development.

It creates the project structure, defines anonymous participant/session identifiers, records or inventories videos, generates multi-label annotations, validates temporal labels, and creates participant-independent splits.

> The legacy still-image ZIP remains useful for the V1 baseline, but MotionBERT experiments require genuine consecutive video frames. Repeating still images does not create temporal evidence.

## V2 research objective

**Primary question:** Does a temporal 3D pose representation improve posture classification, issue recognition, and prediction stability compared with the V1 MediaPipe 2D-landmark Random Forest?

The planned comparison is:

1. V1: single-frame 2D landmarks + Random Forest
2. V2-A: single-frame MotionBERT-derived 3D features
3. V2-B: temporal 3D sequences
4. V2-C: temporal 3D sequences + personal calibration

## 1. Setup

In [12]:
# Uncomment in a fresh environment:
# %pip install -q opencv-python pandas numpy matplotlib seaborn

import hashlib
import json
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
print('OpenCV:', cv2.__version__)

OpenCV: 5.0.0


## 2. Create the V2 project structure

In [13]:
working_directory = Path.cwd()
if working_directory.name == 'notebooks':
    PROJECT_ROOT = working_directory.parent
elif working_directory.name == 'project_v2':
    PROJECT_ROOT = working_directory
else:
    PROJECT_ROOT = working_directory / 'project_v2'

DIRECTORIES = [
    'data/legacy_images', 'data/raw_videos', 'data/processed_videos',
    'data/keypoints_2d', 'data/poses_3d', 'data/annotations',
    'notebooks', 'motionbert', 'models', 'results', 'app',
]
for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)

print('V2 project root:', PROJECT_ROOT.resolve())
for relative_path in DIRECTORIES:
    print(' -', relative_path)

V2 project root: C:\Users\manth\Desktop\project_v2
 - data/legacy_images
 - data/raw_videos
 - data/processed_videos
 - data/keypoints_2d
 - data/poses_3d
 - data/annotations
 - notebooks
 - motionbert
 - models
 - results
 - app


In [14]:
LEGACY_ZIP_NAME = 'Good or Bad Posture v2.v2i.multiclass.zip'
expected_legacy_path = PROJECT_ROOT / 'data' / 'legacy_images' / LEGACY_ZIP_NAME
root_legacy_path = PROJECT_ROOT / LEGACY_ZIP_NAME
if expected_legacy_path.exists():
    print('Legacy V1 dataset found:', expected_legacy_path)
elif root_legacy_path.exists():
    print('Move this file into data/legacy_images before continuing:')
    print(root_legacy_path)
else:
    print('Legacy ZIP not found yet. Expected location:')
    print(expected_legacy_path)

Legacy V1 dataset found: c:\Users\manth\Desktop\project_v2\data\legacy_images\Good or Bad Posture v2.v2i.multiclass.zip


## 3. Data governance and recording protocol

Before recording anyone:

- Obtain informed consent appropriate to your institution and intended publication.
- Use anonymous identifiers such as `P001`; never put a participant's name in filenames.
- Explain how videos will be stored, used, shared, and deleted.
- Do not record audio unless it is genuinely required and explicitly consented to.
- Restrict access to raw identifiable videos. Derived skeletons should still be treated as participant data.
- Allow participants to stop or withdraw without penalty.

### Standard session

Use a fixed camera and chair for each session. The head, shoulders, hips, and knees should remain visible. Record the camera view and setup in `sessions.csv`.

Recommended segments:

1. Comfortable upright calibration — 10 seconds
2. Natural good posture — 25 seconds
3. Forward head — 25 seconds
4. Slouching — 25 seconds
5. Leaning left — 20 seconds
6. Leaning right — 20 seconds
7. Shoulder imbalance — 20 seconds
8. Mixed natural transitions — 40 seconds

Do not instruct participants to hold painful or extreme positions.

## 4. Create annotation templates

In [15]:
ANNOTATION_DIR = PROJECT_ROOT / 'data' / 'annotations'
PARTICIPANTS_FILE = ANNOTATION_DIR / 'participants.csv'
SESSIONS_FILE = ANNOTATION_DIR / 'sessions.csv'
CLIPS_FILE = ANNOTATION_DIR / 'clips.csv'

participant_columns = [
    'participant_id', 'consent_confirmed', 'consent_reference',
    'collection_date', 'split', 'notes',
]
session_columns = [
    'session_id', 'participant_id', 'video_filename', 'camera_view',
    'camera_distance_cm', 'resolution', 'target_fps', 'lighting',
    'chair_description', 'calibration_start_seconds',
    'calibration_end_seconds', 'notes',
]
clip_columns = [
    'clip_id', 'session_id', 'video_filename', 'start_seconds', 'end_seconds',
    'good_posture', 'forward_head', 'slouching', 'lean_left', 'lean_right',
    'shoulder_imbalance', 'transition', 'uncertain', 'annotator_id', 'notes',
]

templates = {
    PARTICIPANTS_FILE: participant_columns,
    SESSIONS_FILE: session_columns,
    CLIPS_FILE: clip_columns,
}
for path, columns in templates.items():
    if not path.exists():
        pd.DataFrame(columns=columns).to_csv(path, index=False)
        print('Created:', path.name)
    else:
        print('Preserved existing:', path.name)

Preserved existing: participants.csv
Preserved existing: sessions.csv
Preserved existing: clips.csv


### Label definitions

| Label | Operational definition |
|---|---|
| `good_posture` | Participant's calibrated upright posture without another issue label |
| `forward_head` | Head displaced forward relative to calibrated shoulders/torso |
| `slouching` | Sustained trunk/spine flexion relative to calibration |
| `lean_left` | Torso deviates left relative to calibrated upright orientation |
| `lean_right` | Torso deviates right relative to calibrated upright orientation |
| `shoulder_imbalance` | Shoulder orientation deviates materially from the calibrated baseline |
| `transition` | Intentional movement between stable postures |
| `uncertain` | View, occlusion, or ambiguity prevents reliable labeling |

Issue labels are multi-label. `good_posture` must not coexist with an issue label in the same interval. Calibration defines a personal baseline; it is not assumed to be a universal clinical ideal.

## 5. Optional standardized webcam recorder

In [16]:
RECORDING_PLAN = [
    ('calibration', 10), ('good_posture', 25), ('forward_head', 25),
    ('slouching', 25), ('lean_left', 20), ('lean_right', 20),
    ('shoulder_imbalance', 20), ('mixed_transitions', 40),
]
display(pd.DataFrame(RECORDING_PLAN, columns=['segment', 'seconds']))

,segment,seconds
0,calibration,10
1,good_posture,25
2,forward_head,25
3,slouching,25
4,lean_left,20
5,lean_right,20
6,shoulder_imbalance,20
7,mixed_transitions,40


In [17]:
def validate_identifier(value, prefix):
    if not re.fullmatch(fr'{prefix}\d{{3}}', value):
        raise ValueError(f'Expected an identifier such as {prefix}001, received {value!r}')

def record_segment(participant_id, session_id, segment_name, duration_seconds,
                   camera_index=0, width=1280, height=720, target_fps=30):
    validate_identifier(participant_id, 'P')
    validate_identifier(session_id, 'S')
    safe_segment = re.sub(r'[^a-z0-9_]+', '_', segment_name.lower()).strip('_')
    output_name = f'{participant_id}_{session_id}_{safe_segment}.mp4'
    output_path = PROJECT_ROOT / 'data' / 'raw_videos' / output_name
    if output_path.exists():
        raise FileExistsError(f'Refusing to overwrite existing recording: {output_path}')

    camera = cv2.VideoCapture(camera_index)
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
    camera.set(cv2.CAP_PROP_FPS, target_fps)
    if not camera.isOpened():
        camera.release()
        raise RuntimeError(f'Could not open camera index {camera_index}')

    actual_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
    actual_fps = camera.get(cv2.CAP_PROP_FPS) or target_fps
    writer = cv2.VideoWriter(
        str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), target_fps,
        (actual_width, actual_height),
    )
    if not writer.isOpened():
        camera.release()
        raise RuntimeError('Could not create the MP4 writer.')

    start = time.perf_counter()
    frames_written = 0
    try:
        while True:
            success, frame = camera.read()
            if not success:
                break
            elapsed = time.perf_counter() - start
            if elapsed >= duration_seconds:
                break
            writer.write(frame)
            frames_written += 1
            preview = frame.copy()
            cv2.putText(preview, f'{segment_name} | {duration_seconds - elapsed:.1f}s',
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.putText(preview, 'Press Q to stop early', (20, 75),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
            cv2.imshow('V2 Data Recorder', preview)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        writer.release()
        camera.release()
        cv2.destroyAllWindows()

    measured_duration = frames_written / target_fps
    print(f'Saved {frames_written} frames ({measured_duration:.1f}s) to {output_path}')
    return {
        'video_filename': output_name, 'frames_written': frames_written,
        'duration_seconds': measured_duration,
        'resolution': f'{actual_width}x{actual_height}',
        'requested_fps': target_fps, 'reported_camera_fps': actual_fps,
    }

To record a segment, remove the leading `#` only after consent and setup checks are complete:

```python
# record_segment('P001', 'S001', 'calibration', 10)
# record_segment('P001', 'S001', 'good_posture', 25)
```

The recorder does not automatically claim labels are correct. Add the recording to `sessions.csv` and review its time intervals in `clips.csv`.

## 6. Optional public sample video (pipeline smoke test only)

Use this free side-view desk video to verify that Notebook 02 can read a video and extract a temporal pose sequence:

**Source:** [Woman Using Laptop at Home — Pexels video 9130469](https://www.pexels.com/video/woman-using-laptop-at-home-9130469/)

Download an MP4 from that page, then edit `DOWNLOADED_SAMPLE` below so it points to the downloaded file. The cell copies it into `data/raw_videos` and records its provenance. A webpage URL is used instead of a hard-coded MP4 URL because Pexels download addresses may expire.

> This external clip is for a pipeline demonstration only. Do not include it in participant-level training, validation, or test metrics: it has no study consent record, participant ID, calibration segment, or research-quality temporal labels.

In [18]:
SAMPLE_PAGE_URL = 'https://www.pexels.com/video/woman-using-laptop-at-home-9130469/'
SAMPLE_CREATOR = 'Dario Fernandez Ruz'

# Change this filename to the exact MP4 name shown in your Downloads folder.
DOWNLOADED_SAMPLE = Path.home() / 'Downloads' / '9130469-uhd_2160_3840_30fps.mp4'
SAMPLE_DESTINATION = (
    PROJECT_ROOT / 'data' / 'raw_videos' / 'DEMO_PEXELS_9130469_side_view.mp4'
)
SAMPLE_SOURCES_FILE = ANNOTATION_DIR / 'external_sample_sources.csv'

if not DOWNLOADED_SAMPLE.exists():
    print('Download the video from:')
    print(SAMPLE_PAGE_URL)
    print('Then change DOWNLOADED_SAMPLE to the exact downloaded MP4 path.')
    print('Current path does not exist:', DOWNLOADED_SAMPLE)
else:
    if SAMPLE_DESTINATION.exists():
        print('Demo video already exists; keeping it unchanged:', SAMPLE_DESTINATION)
    else:
        shutil.copy2(DOWNLOADED_SAMPLE, SAMPLE_DESTINATION)
        print('Copied demo video to:', SAMPLE_DESTINATION.resolve())

    sample_source = pd.DataFrame([{
        'local_filename': SAMPLE_DESTINATION.name,
        'source_page': SAMPLE_PAGE_URL,
        'creator': SAMPLE_CREATOR,
        'provider': 'Pexels',
        'intended_use': 'pipeline_smoke_test_only',
        'exclude_from_model_evaluation': True,
        'retrieved_utc': datetime.now(timezone.utc).isoformat(),
    }])
    if SAMPLE_SOURCES_FILE.exists():
        existing_sources = pd.read_csv(SAMPLE_SOURCES_FILE)
        sample_source = pd.concat([existing_sources, sample_source], ignore_index=True)
        sample_source = sample_source.drop_duplicates('local_filename', keep='last')
    sample_source.to_csv(SAMPLE_SOURCES_FILE, index=False)
    display(sample_source.tail())

Copied demo video to: C:\Users\manth\Desktop\project_v2\data\raw_videos\DEMO_PEXELS_9130469_side_view.mp4


,local_filename,source_page,creator,provider,intended_use,exclude_from_model_evaluation,retrieved_utc
0,DEMO_PEXELS_9130469_side_view.mp4,https://www.pexels.com/video/woman-using-lapto...,Dario Fernandez Ruz,Pexels,pipeline_smoke_test_only,True,2026-09-22T19:33:10.388428+00:00


## 7. Inventory all raw videos

In [19]:
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
RAW_VIDEO_DIR = PROJECT_ROOT / 'data' / 'raw_videos'

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_video(path):
    capture = cv2.VideoCapture(str(path))
    readable = capture.isOpened()
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT)) if readable else 0
    fps = float(capture.get(cv2.CAP_PROP_FPS)) if readable else 0.0
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)) if readable else 0
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)) if readable else 0
    capture.release()
    duration = frame_count / fps if fps > 0 else np.nan
    return {
        'video_filename': path.name, 'relative_path': path.relative_to(PROJECT_ROOT).as_posix(),
        'readable': readable, 'frame_count': frame_count, 'fps': fps,
        'width': width, 'height': height, 'duration_seconds': duration,
        'size_bytes': path.stat().st_size, 'sha256': sha256_file(path),
    }

video_paths = sorted(path for path in RAW_VIDEO_DIR.rglob('*') if path.suffix.lower() in VIDEO_EXTENSIONS)
video_inventory = pd.DataFrame([inspect_video(path) for path in video_paths])
inventory_path = ANNOTATION_DIR / 'video_inventory.csv'
video_inventory.to_csv(inventory_path, index=False)
print('Raw videos:', len(video_inventory))
display(video_inventory)

Raw videos: 1


,video_filename,relative_path,readable,frame_count,fps,width,height,duration_seconds,size_bytes,sha256
0,DEMO_PEXELS_9130469_side_view.mp4,data/raw_videos/DEMO_PEXELS_9130469_side_view.mp4,True,402,30.0,2160,3840,13.4,37268482,ebece8c9ce41011e74618a25d1c2f6580379c203591e90...


## 8. Validate manifests and temporal annotations

In [20]:
LABEL_COLUMNS = [
    'good_posture', 'forward_head', 'slouching', 'lean_left', 'lean_right',
    'shoulder_imbalance', 'transition', 'uncertain',
]

def read_manifest(path, expected_columns):
    frame = pd.read_csv(path)
    missing = set(expected_columns) - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    return frame

participants = read_manifest(PARTICIPANTS_FILE, participant_columns)
sessions = read_manifest(SESSIONS_FILE, session_columns)
clips = read_manifest(CLIPS_FILE, clip_columns)
issues = []

if participants['participant_id'].duplicated().any():
    issues.append('Duplicate participant_id values')
if sessions['session_id'].duplicated().any():
    issues.append('Duplicate session_id values')
if clips['clip_id'].duplicated().any():
    issues.append('Duplicate clip_id values')
if len(sessions) and not sessions['participant_id'].isin(participants['participant_id']).all():
    issues.append('sessions.csv references unknown participants')
if len(clips) and not clips['session_id'].isin(sessions['session_id']).all():
    issues.append('clips.csv references unknown sessions')
if len(clips):
    for column in LABEL_COLUMNS:
        if not clips[column].fillna(0).isin([0, 1]).all():
            issues.append(f'{column} contains values other than 0 or 1')
    if not (clips['start_seconds'] < clips['end_seconds']).all():
        issues.append('One or more clip intervals have start_seconds >= end_seconds')
    issue_labels = ['forward_head', 'slouching', 'lean_left', 'lean_right', 'shoulder_imbalance']
    conflict = clips['good_posture'].eq(1) & clips[issue_labels].fillna(0).eq(1).any(axis=1)
    if conflict.any():
        issues.append(f'{int(conflict.sum())} intervals combine good_posture with issue labels')
    unlabeled = clips[LABEL_COLUMNS].fillna(0).sum(axis=1).eq(0)
    if unlabeled.any():
        issues.append(f'{int(unlabeled.sum())} intervals have no label')

if issues:
    print('VALIDATION ISSUES:')
    for issue in issues:
        print(' -', issue)
else:
    print('Manifest validation passed. Empty templates are expected before collection.')

Manifest validation passed. Empty templates are expected before collection.


## 9. Assign participant-independent splits
Run this only after participant rows exist. The mapping is deterministic for the fixed seed. Once a participant enters the test set, freeze that assignment and do not move them based on model performance.

In [21]:
def assign_participant_splits(participant_frame, seed=SEED):
    result = participant_frame.copy()
    identifiers = result['participant_id'].dropna().astype(str).unique()
    if len(identifiers) < 6:
        print('At least 6 participants are required to create non-empty train/valid/test groups.')
        return result
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(sorted(identifiers))
    test_count = max(1, round(0.15 * len(shuffled)))
    valid_count = max(1, round(0.15 * len(shuffled)))
    mapping = {}
    for participant_id in shuffled[:test_count]:
        mapping[participant_id] = 'test'
    for participant_id in shuffled[test_count:test_count + valid_count]:
        mapping[participant_id] = 'valid'
    for participant_id in shuffled[test_count + valid_count:]:
        mapping[participant_id] = 'train'
    result['split'] = result['participant_id'].map(mapping)
    return result

if len(participants):
    proposed_participants = assign_participant_splits(participants)
    display(proposed_participants[['participant_id', 'split']])
    print('Review the proposed mapping, then explicitly save it with:')
    print("proposed_participants.to_csv(PARTICIPANTS_FILE, index=False)")
else:
    print('No participants recorded yet; split assignment was skipped.')

No participants recorded yet; split assignment was skipped.


## 10. Dataset coverage report

In [22]:
if len(clips):
    clips = clips.copy()
    clips['duration_seconds'] = clips['end_seconds'] - clips['start_seconds']
    duration_by_label = pd.Series({
        label: clips.loc[clips[label].eq(1), 'duration_seconds'].sum()
        for label in LABEL_COLUMNS
    }).sort_values(ascending=False)
    display(duration_by_label.rename('annotated_seconds').to_frame())
    plt.figure(figsize=(10, 4))
    sns.barplot(x=duration_by_label.index, y=duration_by_label.values, color='steelblue')
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('Annotated seconds')
    plt.title('Temporal label coverage')
    plt.tight_layout()
    plt.show()
else:
    print('Coverage report will become available after clips.csv is annotated.')

Coverage report will become available after clips.csv is annotated.


## 11. Save the V2 dataset protocol record

In [23]:
protocol = {
    'project': 'Temporal 3D Sitting Posture Detection V2',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'seed': SEED,
    'target_resolution': '1280x720',
    'target_fps': 30,
    'motionbert_joint_format': 'Human3.6M 17-joint',
    'source_2d_format': 'MediaPipe 33-landmark',
    'split_unit': 'participant',
    'labels': LABEL_COLUMNS,
    'recording_plan': [{'segment': name, 'seconds': seconds} for name, seconds in RECORDING_PLAN],
    'raw_video_count': int(len(video_inventory)),
    'participant_count': int(len(participants)),
    'session_count': int(len(sessions)),
    'annotated_interval_count': int(len(clips)),
}
protocol_path = PROJECT_ROOT / 'results' / 'dataset_protocol.json'
with protocol_path.open('w') as file:
    json.dump(protocol, file, indent=2)
print('Saved protocol:', protocol_path.resolve())
display(pd.Series(protocol, name='value').to_frame())

Saved protocol: C:\Users\manth\Desktop\project_v2\results\dataset_protocol.json


,value
project,Temporal 3D Sitting Posture Detection V2
created_utc,2026-09-22T19:33:26.911227+00:00
seed,42
target_resolution,1280x720
target_fps,30
motionbert_joint_format,Human3.6M 17-joint
source_2d_format,MediaPipe 33-landmark
split_unit,participant
labels,"[good_posture, forward_head, slouching, lean_l..."
recording_plan,"[{'segment': 'calibration', 'seconds': 10}, {'..."


## Completion criteria for Notebook 01

Notebook 01 is complete when:

- the legacy ZIP is stored under `data/legacy_images`;
- consent and storage procedures are defined;
- every participant has an anonymous identifier;
- recordings use consistent session and filename conventions;
- video inventory reports readable files, FPS, duration, and resolution;
- temporal labels follow the operational definitions;
- manifest validation has no unresolved errors; and
- participant-independent split assignments are frozen.

### Next notebook
**V2 Notebook 02: Temporal 2D Keypoint Extraction** will run MediaPipe in video mode, preserve confidence values and timestamps, map MediaPipe's 33 landmarks into MotionBERT's 17-joint Human3.6M representation, visualize the mapping, and export fixed-length temporal windows.